## 实现正式LLM架构


In [29]:
GPT_CONFIG_124M = {
    'vocab_size': 50257,  # 字典大小，BPE 分词器使用的
    'context_length': 1024,  # 上下文长度，模型所能处理的最大输入 token 数
    'emb_dim': 768,  # 嵌入维度，将每个 token 转为 768维度的向量
    'n_heads': 12,  # number of attention heads
    'n_layers': 12,  # number of layers，transformer 模块的层数
    'drop_rate': 0.1,  # dropout rate，0.1表示丢失 10% 的隐藏单元，用于防止过拟合
    'qkv_bias': False,  # query-key-value bias，用于决定是否在多头注意力的查询、键、值的线性层中加入偏置向量
    # 最初会禁用该选项，以遵循现代大语言模型的标准，之后在第 6 章加载 OpenAI 预训练的 GPT-2 权重时再重新考虑该设置。
}

In [30]:
import torch
import torch.nn as nn

In [31]:
import tiktoken

tokenizer = tiktoken.get_encoding('gpt2')

In [32]:
# Layer Normalization 模块
# 把每个 token 的特征向量归一化，再进行可学习的缩放和平移
# 以提高训练的稳定性和模型的表达能力。
class LayerNorm(nn.Module):
    # emb_dim 每个 token的向量维度，例如 768
    def __init__(self, emb_dim):
        super().__init__()
        # 很小的常数，防止方差为 0 时出现除 0
        self.eps = 1e-5
        # 两个长度为 emb_dim 的可训练参数，scale是缩放参数，shift是平移参数
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # dim=-1，沿最后一个维度计算均值；对于 2*4*768的输入，会生成一个 2*4*1的输出
        mean = x.mean(dim=-1, keepdim=True)
        # 沿最后一个维度计算反差
        # unbiased=False表示使用总体方差 var = Σ(xᵢ - mean)² / N，而不是样本方差中的 / (N - 1)
        var = x.var(dim=-1, unbiased=False, keepdim=True)
        # 进行标准化
        #              x - 均值
        # norm_x = ─────────────
        #           √(方差 + eps)
        # 归一化后，每个 token 向量大致满足：
        #       均值 ≈ 0
        #       方差 ≈ 1
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        # 对归一化结果进行可学习的缩放和平移
        return self.scale * norm_x + self.shift

In [33]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [34]:
# FeedForward 模块是一个小型神经网络，由两个线性层和一个 GELU 激活函数组成。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            # 这是一个全连接层，把每个 token 的特征向量从 emb_dim 维扩展到 4 × emb_dim 维。
            # 假设输入形状为：[batch_size, token数量, 768]
            # 经过线性层后：  [batch_size, token数量, 3072]
            # 前两个维度保持不变，只转换最后一个特征维度。
            # 内部计算为：y = xWᵀ + b
            # 参数形状是：linear.weight.shape  # [3072, 768]
            #           linear.bias.shape    # [3072]
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),  # 扩展
            GELU(),                                         # 非线性变换
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'])   # 压缩回原维度
        )

    def forward(self, x):
        return self.layers(x)

In [35]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, 'd_out must be divisible by num_heads'
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [36]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            num_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'],
            qkv_bias=cfg['qkv_bias'],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [37]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])

        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(torch.arange(seq_len, device=tok_embeds.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [38]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)


In [39]:
def generate_text_simple(model, idx, max_new_token, context_size):
    for _ in range(max_new_token):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [40]:
encoded = tokenizer.encode('Hello, I am')
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

model.eval()
out = generate_text_simple(model=model, idx=encoded_tensor, max_new_token=6,
                           context_size=GPT_CONFIG_124M['context_length'])

print('output: ', out)
print('output length: ', len(out[0]))

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

output:  tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
output length:  10
Hello, I am Featureiman Byeswickattribute argue
